<a href="https://colab.research.google.com/github/yeminaung-web/AI_STUDY/blob/main/UCI_HAR/UCI_HAR(CBAM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import urllib.request
import zipfile
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:

# ============================================================
# 1. CONFIGURATION & REPRODUCIBILITY
# ============================================================
SEED = 42
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-3
MODEL_PATH = "best_cbam_bilstm_uci_har.pt"
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/human+activity+recognition+using+smartphones/UCI HAR Dataset"

ACTIVITY_LABELS = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING",
]

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [7]:

# ============================================================
# 2. DOWNLOAD & LOAD UCI-HAR INERTIAL SIGNALS
# ============================================================
if not os.path.exists(DATASET_PATH):
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip"
    urllib.request.urlretrieve(url, "uci_har.zip")
    with zipfile.ZipFile("uci_har.zip", "r") as zf:
        zf.extractall(".")

SIGNALS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]

def load_split(subset):
    signals_data = []
    for signal in SIGNALS:
        file_path = os.path.join(DATASET_PATH, subset, "Inertial Signals", f"{signal}_{subset}.txt")
        signals_data.append(np.loadtxt(file_path))
    # Transpose from (9, samples, 128) -> (samples, 128, 9)
    X = np.transpose(np.array(signals_data), (1, 2, 0)).astype(np.float32)
    y_path = os.path.join(DATASET_PATH, subset, f"y_{subset}.txt")
    y = (np.loadtxt(y_path, dtype=np.int64) - 1)  # Convert labels 1..6 -> 0..5
    return X, y

X_train_full, y_train_full = load_split("train")
X_test, y_test = load_split("test")

# Train / Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=SEED, stratify=y_train_full
)

# Leakage-free StandardScaler (fit ONLY on X_train)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, 9)).reshape(X_train.shape)
X_val = scaler.transform(X_val.reshape(-1, 9)).reshape(X_val.shape)
X_test = scaler.transform(X_test.reshape(-1, 9)).reshape(X_test.shape)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val), torch.tensor(y_val)),
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
    batch_size=BATCH_SIZE,
    shuffle=False,
)


In [8]:

# ============================================================
# 3. 1D CBAM + BiLSTM ARCHITECTURE
# ============================================================
class ChannelAttention1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.shared_mlp = nn.Sequential(
            nn.Conv1d(channels, hidden, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv1d(hidden, channels, kernel_size=1, bias=False),
        )

    def forward(self, x):
        # x: (batch, channels, time)
        avg_out = self.shared_mlp(F.adaptive_avg_pool1d(x, 1))
        max_out = self.shared_mlp(F.adaptive_max_pool1d(x, 1))
        scale = torch.sigmoid(avg_out + max_out)
        return x * scale


class TemporalAttention1D(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv1d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)

    def forward(self, x):
        # x: (batch, channels, time)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)  # (batch, 2, time)
        scale = torch.sigmoid(self.conv(concat))       # (batch, 1, time)
        return x * scale


class CBAMBlock1D(nn.Module):
    def __init__(self, channels, reduction=8, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention1D(channels, reduction=reduction)
        self.temporal_att = TemporalAttention1D(kernel_size=kernel_size)

    def forward(self, x):
        x = self.channel_att(x)
        x = self.temporal_att(x)
        return x


class CBAMBiLSTM(nn.Module):
    def __init__(self, in_channels=9, num_classes=6):
        super().__init__()
        # Block 1: Conv1D + CBAM
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
        )
        self.cbam1 = CBAMBlock1D(channels=64, reduction=8, kernel_size=7)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout(0.30)

        # Block 2: Conv1D + CBAM
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
        )
        self.cbam2 = CBAMBlock1D(channels=128, reduction=8, kernel_size=7)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout(0.30)

        # BiLSTM Sequence Core
        self.bilstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            batch_first=True,
            bidirectional=True,
        )
        self.drop3 = nn.Dropout(0.30)

        # Dense Classifier
        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        # (batch, 128, 9) -> (batch, 9, 128) for Conv1d
        x = x.permute(0, 2, 1)

        x = self.cbam1(self.conv1(x))
        x = self.drop1(self.pool1(x))  # (batch, 64, 64)

        x = self.cbam2(self.conv2(x))
        x = self.drop2(self.pool2(x))  # (batch, 128, 32)

        # (batch, 128, 32) -> (batch, 32, 128) for LSTM
        x = x.permute(0, 2, 1)

        _, (h_n, _) = self.bilstm(x)
        x = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (batch, 256)
        x = self.drop3(x)

        logits = self.classifier(x)
        return logits


model = CBAMBiLSTM(in_channels=9, num_classes=6).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")


Total params: 330,402 | Trainable: 330,402


In [9]:

# ============================================================
# 4. TRAINING LOOP (WITH LR SCHEDULER & EARLY STOPPING)
# ============================================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

best_val_loss = float("inf")
best_epoch = 0
early_stop_patience = 15
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        train_correct += (logits.argmax(dim=1) == yb).sum().item()
        train_total += xb.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss += loss.item() * xb.size(0)
            val_correct += (logits.argmax(dim=1) == yb).sum().item()
            val_total += xb.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f} | LR: {current_lr:.2e}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stop_patience:
            print(f"Early stopping at Epoch {epoch}. Best Epoch: {best_epoch} (Val Loss: {best_val_loss:.5f})")
            break


Epoch 001/100 | Train Loss: 0.7656, Acc: 0.6618 | Val Loss: 0.3512, Acc: 0.8348 | LR: 1.00e-03
Epoch 002/100 | Train Loss: 0.2830, Acc: 0.8947 | Val Loss: 0.1460, Acc: 0.9531 | LR: 1.00e-03
Epoch 003/100 | Train Loss: 0.1696, Acc: 0.9417 | Val Loss: 0.1147, Acc: 0.9613 | LR: 1.00e-03
Epoch 004/100 | Train Loss: 0.1453, Acc: 0.9408 | Val Loss: 0.2209, Acc: 0.9137 | LR: 1.00e-03
Epoch 005/100 | Train Loss: 0.1497, Acc: 0.9381 | Val Loss: 0.0971, Acc: 0.9633 | LR: 1.00e-03
Epoch 006/100 | Train Loss: 0.1298, Acc: 0.9481 | Val Loss: 0.0993, Acc: 0.9599 | LR: 1.00e-03
Epoch 007/100 | Train Loss: 0.1380, Acc: 0.9424 | Val Loss: 0.1164, Acc: 0.9585 | LR: 1.00e-03
Epoch 008/100 | Train Loss: 0.1342, Acc: 0.9432 | Val Loss: 0.1462, Acc: 0.9456 | LR: 1.00e-03
Epoch 009/100 | Train Loss: 0.1360, Acc: 0.9493 | Val Loss: 0.0972, Acc: 0.9585 | LR: 1.00e-03
Epoch 010/100 | Train Loss: 0.1236, Acc: 0.9493 | Val Loss: 0.0988, Acc: 0.9572 | LR: 1.00e-03
Epoch 011/100 | Train Loss: 0.1137, Acc: 0.9510 | 

In [10]:

# ============================================================
# 5. TEST EVALUATION
# ============================================================
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

test_loss, test_total = 0.0, 0
all_preds, all_targets = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)

        test_loss += loss.item() * xb.size(0)
        test_total += xb.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_targets.extend(yb.cpu().numpy())

test_loss /= test_total
y_pred = np.array(all_preds)
y_true = np.array(all_targets)

acc = accuracy_score(y_true, y_pred)
m_prec, m_rec, m_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
_, _, w_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")

print("\n" + "=" * 70)
print("PYTORCH CBAM + BiLSTM TEST RESULTS")
print("=" * 70)
print(f"Test Loss       : {test_loss:.4f}")
print(f"Accuracy        : {acc:.4f}")
print(f"Macro Precision : {m_prec:.4f}")
print(f"Macro Recall    : {m_rec:.4f}")
print(f"Macro F1        : {m_f1:.4f}")
print(f"Weighted F1     : {w_f1:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=ACTIVITY_LABELS, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))



PYTORCH CBAM + BiLSTM TEST RESULTS
Test Loss       : 0.2277
Accuracy        : 0.9589
Macro Precision : 0.9592
Macro Recall    : 0.9595
Macro F1        : 0.9590
Weighted F1     : 0.9588

Classification Report:

                    precision    recall  f1-score   support

           WALKING     0.9980    0.9899    0.9939       496
  WALKING_UPSTAIRS     0.9890    0.9512    0.9697       471
WALKING_DOWNSTAIRS     0.9437    0.9976    0.9699       420
           SITTING     0.9234    0.8839    0.9032       491
          STANDING     0.9119    0.9342    0.9229       532
            LAYING     0.9890    1.0000    0.9944       537

          accuracy                         0.9589      2947
         macro avg     0.9592    0.9595    0.9590      2947
      weighted avg     0.9592    0.9589    0.9588      2947

Confusion Matrix:
 [[491   1   4   0   0   0]
 [  1 448  21   1   0   0]
 [  0   1 419   0   0   0]
 [  0   3   0 434  48   6]
 [  0   0   0  35 497   0]
 [  0   0   0   0   0 537]]
